# 🦇 MORRIGAN — Phase 1: Complete Data Pipeline
## Colab Notebook → Downloads 73 datasets → Converts → Filters → Samples → Restylizes

**Run each cell in order.** The full pipeline takes ~6-10 hours (mostly restylization API calls).

**What this notebook does:**
1. Downloads all 73 datasets from HuggingFace
2. Converts every format to unified ShareGPT JSONL
3. Filters for quality, English, relevance
4. Samples 5,000 entries by strategic category mix
5. Restylizes all assistant responses as Morrigan via Claude API
6. Saves final training files to Google Drive for transfer to RunPod

**Requirements:**
- HuggingFace account (free) — for gated dataset access
- Anthropic API key — for restylization ($25-35)
- Google Drive — for saving outputs


## 1. Setup & Install

In [ ]:
# Install dependencies
!pip install -q datasets transformers anthropic tqdm huggingface_hub

# Mount Google Drive for saving outputs
from google.colab import drive
drive.mount('/content/drive')

# Create project directories
import os
BASE = "/content/morrigan-dataset"
for d in ["raw", "processed", "training"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

# Also create output dir in Drive
DRIVE_OUT = "/content/drive/MyDrive/morrigan-training"
os.makedirs(DRIVE_OUT, exist_ok=True)

print("✅ Setup complete")


## 2. Login to HuggingFace & Set API Key

In [ ]:
# Login to HuggingFace (paste your token when prompted)
from huggingface_hub import login
login()

# Set your Anthropic API key
import os
os.environ["ANTHROPIC_API_KEY"] = ""  # <-- PASTE YOUR KEY HERE THEN RUN
# Example: os.environ["ANTHROPIC_API_KEY"] = "sk-ant-api03-..."

assert os.environ.get("ANTHROPIC_API_KEY"), "⚠️ Set your API key above!"
print("✅ Logged in and API key set")


## 3. Download All 73 Datasets

This downloads every dataset to `/content/morrigan-dataset/raw/`.
Gated datasets that need approval are commented out — uncomment when approved.
Some datasets may fail (removed, renamed) — the script continues past failures.

**This cell takes 30-60 minutes depending on connection speed.**


In [ ]:
from datasets import load_dataset
import os, traceback

RAW = "/content/morrigan-dataset/raw"

# ============================================================
# COMPLETE DATASET REGISTRY — ALL 73 DATASETS
# Format: "local_name": ("hf_path", "subset_or_None")
# ============================================================
DATASETS = {
    # ═══ CATEGORY A: Emotional Support (10 datasets) ═══
    "empathetic_dialogues":         ("facebook/empathetic_dialogues", None),
    "empathetic_dialogues_sharegpt":("Estwld/empathetic_dialogues_llm", None),
    "esconv":                       ("thu-coai/esconv", None),
    "augesc":                       ("thu-coai/augesc", None),
    "mentalchat16k":                ("ShenLab/MentalChat16K", None),
    "mental_health_counseling":     ("Amod/mental_health_counseling_conversations", None),
    "mental_health_chatbot":        ("heliosbrahma/mental_health_chatbot_dataset", None),
    "annomi":                       ("to-be/annomi-motivational-interviewing-therapy-conversations", None),
    "therapy_combined":             ("IINOVAII/therapy-conversations-combined", None),
    "empathetic_counseling":        ("mpingale/mental-health-chat-dataset", None),

    # ═══ CATEGORY B: Persona-Grounded (6 datasets) ═══
    "persona_chat":                 ("AlekseyKorshuk/persona-chat", None),
    "synthetic_persona_chat":       ("google/Synthetic-Persona-Chat", None),
    "persona_based_chat_64k":       ("nazlicanto/persona-based-chat", None),
    "opencharacter":                ("xywang1/OpenCharacter", None),
    "roleplay_5k":                  ("hieunguyenminh/roleplay", None),
    "roleplay_instructions":        ("iamketan25/roleplay-instructions-dataset", None),

    # ═══ CATEGORY C: Social & Daily Life (8 datasets) ═══
    "soda":                         ("allenai/soda", None),
    "daily_dialog":                 ("li2017dailydialog/daily_dialog", None),
    "daily_dialog_sharegpt":        ("agentlans/li2017dailydialog", None),
    "prosocial_dialog":             ("allenai/prosocial-dialog", None),
    "blended_skill_talk":           ("ParlAI/blended_skill_talk", None),
    "soc":                          ("HuggingFaceTB/everyday-conversations-llama3.1-2k", None),
    "dialogsum":                    ("knkarthick/dialogsum", None),
    "flirty_dialogue":              ("CanadianGamer/Flirty-Dialouge", None),

    # ═══ CATEGORY D: Roleplay & Creative (10 datasets) ═══
    "pippa":                        ("PygmalionAI/PIPPA", None),
    "pippa_sharegpt":               ("kingbri/PIPPA-shareGPT", None),
    "limarp":                       ("grimulkan/LimaRP-augmented", None),
    "claude_multiround_30k":        ("Norquinal/claude_multiround_chat_30k", None),
    "claude_multiround_1k":         ("Norquinal/claude_multiround_chat_1k", None),
    "creative_writing":             ("Nitral-AI/Creative_Writing-ShareGPT", None),
    "roleplay_io":                  ("AlekseyKorshuk/roleplay-io", None),
    "bluemoon":                     ("Squish42/bluemoon-fandom-1-1-rp-cleaned", None),
    "chatgpt_writing_prompts":      ("Gryphe/ChatGPT-4o-Writing-Prompts", None),
    "claude_writing_fixed":         ("anthracite-org/nopm_claude_writing_fixed", None),

    # ═══ CATEGORY E: Real Conversation Logs (5 datasets) ═══
    "wildchat_nontoxic":            ("yuntian-deng/WildChat-1M-Full", None),
    "sharegpt_vicuna":              ("anon8231489123/ShareGPT_Vicuna_unfiltered", None),
    "realtalk":                     ("scottgeng00/realtalk", None),
    "everyday_conversations":       ("HuggingFaceTB/everyday-conversations-llama3.1-2k", None),
    "cai_harmless":                 ("HuggingFaceH4/cai-conversation-harmless", None),

    # ═══ CATEGORY F: Instruction & Preference (6 datasets) ═══
    "counsel_chat":                 ("nbertagnolli/counsel-chat", None),
    "oasst1":                       ("OpenAssistant/oasst1", None),
    "ultrachat_200k":               ("HuggingFaceH4/ultrachat_200k", None),
    "capybara":                     ("LDJnr/Capybara", None),
    "airoboros":                     ("jondurbin/airoboros-gpt4-1.4.1", None),
    "samantha_data":                ("cognitivecomputations/samantha-data", None),

    # ═══ CATEGORY G: Preference / DPO (8 datasets) ═══
    "hh_rlhf":                      ("Anthropic/hh-rlhf", None),
    "magpie_ultra":                 ("Magpie-Align/Magpie-Ultra-v0.1", None),
    "magpie_pro_dpo":               ("Magpie-Align/Magpie-Pro-DPO-100K-v0.1", None),
    "llm_chat_preference":          ("argilla/llm-chat-preference", None),
    "psych8k":                      ("EmoCareAI/Psych8k", None),
    "multi_turn_counsel":           ("Psychotherapy-LLM/PsychoCounsel", None),
    "reflections_peer_counseling":  ("emoneil/reflections-in-peer-counseling", None),
    "sentiment_analysis":           ("Sp1786/multiclass-sentiment-analysis-dataset", None),

    # ═══ CATEGORY H: NEW GOLDMINE — Restricted/Research (10 datasets) ═══
    "rizz_corpus":                  ("the-rizz/the-rizz-corpus", None),
    "kajiwoto":                     ("FarisHijazi/kajiwoto.ai-chat", None),
    "psychocounsel_preference":     ("Psychotherapy-LLM/PsychoCounsel-Preference", None),
    "human_like_dpo":               ("HumanLLMs/Human-Like-DPO-Dataset", None),
    "flirty_or_not":                ("ieuniversity/flirty_or_not", None),
    "toxic_chat":                   ("lmsys/toxic-chat", "toxicchat0124"),
    "psychotherapy":                ("entfane/psychotherapy", None),
    "chat_sentiment":               ("yuncongli/chat-sentiment-analysis", None),

    # ═══ GATED — Uncomment when approved ═══
    # "wildchat_full":              ("yuntian-deng/WildChat-4.8M-Full", None),
    # "lmsys_chat_1m":             ("lmsys/lmsys-chat-1m", None),
    # "chatbot_arena":             ("lmsys/chatbot_arena_conversations", None),
}

print(f"📦 Downloading {len(DATASETS)} datasets...")
print(f"   Saving to: {RAW}")
print()

success = 0
failed = []

for name, (path, subset) in DATASETS.items():
    save_path = os.path.join(RAW, name)
    if os.path.exists(save_path):
        print(f"  ⏭️  {name} (already exists)")
        success += 1
        continue
    try:
        print(f"  ⬇️  {name} ({path})...", end=" ", flush=True)
        if subset:
            ds = load_dataset(path, subset, trust_remote_code=True)
        else:
            ds = load_dataset(path, trust_remote_code=True)
        ds.save_to_disk(save_path)
        print("✅")
        success += 1
    except Exception as e:
        err = str(e)[:80]
        print(f"❌ {err}")
        failed.append((name, err))

print(f"\n{'='*60}")
print(f"✅ Downloaded: {success}/{len(DATASETS)}")
if failed:
    print(f"❌ Failed: {len(failed)}")
    for name, err in failed:
        print(f"   - {name}: {err}")
    print("\n(Failed datasets are often gated or renamed. The pipeline")
    print(" continues with whatever downloaded successfully.)")


## 4. Universal Converter

Converts ALL downloaded datasets into one unified ShareGPT JSONL format.
Handles 10+ different format types automatically.

**This cell takes 5-20 minutes depending on dataset sizes.**


In [ ]:
import json, os, re, sys
from pathlib import Path
from datasets import load_from_disk
from collections import Counter

RAW = "/content/morrigan-dataset/raw"
MASTER_OUT = "/content/morrigan-dataset/processed/master.jsonl"
DPO_OUT = "/content/morrigan-dataset/processed/master_dpo.jsonl"

# ============================================================
# CONVERTER FUNCTIONS
# ============================================================
def add_entry(entries, conversations, source, category="general"):
    roles = [c["role"] for c in conversations if c["role"] in ("user", "assistant")]
    if len(roles) < 2:
        return
    conversations = [c for c in conversations if c.get("content", "").strip()]
    if len(conversations) < 2:
        return
    turns = sum(1 for c in conversations if c["role"] in ("user", "assistant"))
    entries.append({
        "conversations": conversations,
        "source": source,
        "category": category,
        "turns": turns,
    })

def convert_sharegpt(ds, source, category="general"):
    entries = []
    for row in ds:
        convos = row.get("conversations", row.get("messages", []))
        if isinstance(convos, str):
            try: convos = json.loads(convos)
            except: continue
        converted = []
        for turn in convos:
            if not isinstance(turn, dict): continue
            role_raw = turn.get("from", turn.get("role", ""))
            content = turn.get("value", turn.get("content", ""))
            if role_raw in ("human", "user", "Human"):
                converted.append({"role": "user", "content": str(content)})
            elif role_raw in ("gpt", "assistant", "Assistant", "bot", "model"):
                converted.append({"role": "assistant", "content": str(content)})
            elif role_raw in ("system", "System"):
                converted.append({"role": "system", "content": str(content)})
        add_entry(entries, converted, source, category)
    return entries

def convert_dialog_list(ds, source, category="social"):
    entries = []
    for row in ds:
        dialog = row.get("dialogue", row.get("dialog", row.get("utterances",
                 row.get("conversation", []))))
        if isinstance(dialog, str):
            dialog = dialog.split("\n")
        if not isinstance(dialog, list) or len(dialog) < 2: continue
        converted = []
        for i, msg in enumerate(dialog):
            if isinstance(msg, str):
                msg = msg.strip()
                if not msg: continue
                role = "user" if i % 2 == 0 else "assistant"
                converted.append({"role": role, "content": msg})
        add_entry(entries, converted, source, category)
    return entries

def convert_qa(ds, source, category="support"):
    entries = []
    for row in ds:
        user_msg = (row.get("Context", "") or row.get("context", "") or
                    row.get("question", "") or row.get("Question", "") or
                    row.get("input", "") or row.get("instruction", "") or
                    row.get("prompt", "") or "")
        asst_msg = (row.get("Response", "") or row.get("response", "") or
                    row.get("answer", "") or row.get("Answer", "") or
                    row.get("output", "") or "")
        if user_msg and asst_msg:
            converted = [
                {"role": "user", "content": str(user_msg).strip()},
                {"role": "assistant", "content": str(asst_msg).strip()},
            ]
            add_entry(entries, converted, source, category)
    return entries

def convert_structured_dialog(ds, source, category="support"):
    entries = []
    for row in ds:
        dialog = row.get("dialog", row.get("conversation",
                 row.get("dialogue", [])))
        if isinstance(dialog, str):
            try: dialog = json.loads(dialog)
            except: continue
        if not isinstance(dialog, list): continue
        converted = []
        for turn in dialog:
            if isinstance(turn, dict):
                speaker = turn.get("speaker", turn.get("role",
                          turn.get("from", "")))
                content = turn.get("content", turn.get("text",
                          turn.get("value", "")))
                if not content: continue
                if speaker in ("usr","user","seeker","human","client","patient","Human"):
                    converted.append({"role": "user", "content": str(content)})
                elif speaker in ("sys","assistant","supporter","gpt","bot",
                                 "therapist","counselor","model"):
                    converted.append({"role": "assistant", "content": str(content)})
            elif isinstance(turn, str):
                role = "user" if len(converted) % 2 == 0 else "assistant"
                converted.append({"role": role, "content": turn})
        add_entry(entries, converted, source, category)
    return entries

def convert_pippa(ds, source="pippa", category="roleplay"):
    entries = []
    for row in ds:
        messages = row.get("conversation", row.get("messages", []))
        bot_desc = row.get("bot_description", "")
        converted = []
        if bot_desc:
            converted.append({"role": "system", "content": str(bot_desc)})
        for msg in messages:
            if isinstance(msg, dict):
                is_human = msg.get("is_human", False)
                content = msg.get("message", msg.get("content", ""))
                if content:
                    role = "user" if is_human else "assistant"
                    converted.append({"role": role, "content": str(content)})
        add_entry(entries, converted, source, category)
    return entries

def convert_persona_chat(ds, source="persona_chat", category="persona"):
    entries = []
    for row in ds:
        utterances = row.get("utterances", row.get("dialogue", []))
        persona = row.get("personality", row.get("persona", []))
        if isinstance(persona, list): persona_text = " ".join(persona)
        else: persona_text = str(persona) if persona else ""
        converted = []
        if persona_text:
            converted.append({"role": "system", "content": persona_text})
        if isinstance(utterances, list):
            if utterances and isinstance(utterances[0], dict):
                for utt in utterances:
                    history = utt.get("history", [])
                    for i, msg in enumerate(history):
                        role = "user" if i % 2 == 0 else "assistant"
                        converted.append({"role": role, "content": str(msg)})
                    candidates = utt.get("candidates", [])
                    if candidates:
                        converted.append({"role": "assistant",
                                         "content": str(candidates[-1])})
            else:
                for i, msg in enumerate(utterances):
                    role = "user" if i % 2 == 0 else "assistant"
                    if isinstance(msg, str):
                        converted.append({"role": role, "content": msg})
        add_entry(entries, converted, source, category)
    return entries

def convert_wildchat(ds, source="wildchat", category="real_user"):
    entries = []
    for row in ds:
        conversation = row.get("conversation", row.get("messages", []))
        if isinstance(conversation, str):
            try: conversation = json.loads(conversation)
            except: continue
        converted = []
        for msg in conversation:
            if not isinstance(msg, dict): continue
            role = msg.get("role", "")
            content = msg.get("content", "")
            if role in ("user", "assistant", "system") and content:
                converted.append({"role": role, "content": str(content)})
        add_entry(entries, converted, source, category)
    return entries

def convert_hh_rlhf(ds, source="hh_rlhf", category="preference"):
    entries = []
    for row in ds:
        chosen = row.get("chosen", "")
        def parse_hh(text):
            turns = []
            parts = re.split(r'\n\n(Human|Assistant): ', str(text))
            role = None
            for part in parts:
                if part == "Human": role = "user"
                elif part == "Assistant": role = "assistant"
                elif role and part.strip():
                    turns.append({"role": role, "content": part.strip()})
            return turns
        chosen_turns = parse_hh(chosen)
        if chosen_turns:
            add_entry(entries, chosen_turns, source, category)
    return entries

def convert_rizz(ds, source="rizz_corpus", category="dating"):
    entries = []
    for row in ds:
        text = row.get("text", "")
        sys_match = re.search(r'<<SYS>>(.*?)<</SYS>>', str(text), re.DOTALL)
        sys_content = sys_match.group(1).strip() if sys_match else ""
        user_msgs = re.findall(r'\[INST\]\s*(.*?)\s*\[/INST\]', str(text))
        asst_msgs = re.findall(r'\[/INST\](.*?)</s>', str(text))
        if user_msgs and asst_msgs:
            converted = []
            if sys_content:
                converted.append({"role": "system", "content": sys_content})
            for u, a in zip(user_msgs, asst_msgs):
                u, a = u.strip(), a.strip()
                if u and a:
                    converted.append({"role": "user", "content": u})
                    converted.append({"role": "assistant", "content": a})
            add_entry(entries, converted, source, category)
    return entries

def convert_dpo(ds, source, category="preference"):
    entries = []
    dpo_entries = []
    for row in ds:
        prompt = row.get("prompt", row.get("instruction",
                 row.get("question", "")))
        chosen = row.get("chosen", row.get("chosen_response", ""))
        rejected = row.get("rejected", row.get("rejected_response", ""))
        if isinstance(chosen, list):
            chosen = " ".join(c.get("content", str(c)) for c in chosen)
        if isinstance(rejected, list):
            rejected = " ".join(c.get("content", str(c)) for c in rejected)
        if prompt and chosen:
            converted = [
                {"role": "user", "content": str(prompt).strip()},
                {"role": "assistant", "content": str(chosen).strip()},
            ]
            add_entry(entries, converted, source, category)
        if prompt and chosen and rejected:
            dpo_entries.append({
                "prompt": str(prompt).strip(),
                "chosen": str(chosen).strip(),
                "rejected": str(rejected).strip(),
                "source": source,
            })
    return entries, dpo_entries

# ============================================================
# DATASET → CONVERTER MAPPING
# ============================================================
REGISTRY = {
    # Category A: Emotional Support
    "empathetic_dialogues":         ("qa",              "emotional"),
    "empathetic_dialogues_sharegpt":("sharegpt",        "emotional"),
    "esconv":                       ("structured_dialog","emotional"),
    "augesc":                       ("structured_dialog","emotional"),
    "mentalchat16k":                ("qa",              "therapy"),
    "mental_health_counseling":     ("qa",              "therapy"),
    "mental_health_chatbot":        ("qa",              "therapy"),
    "annomi":                       ("sharegpt",        "therapy"),
    "therapy_combined":             ("qa",              "therapy"),
    "empathetic_counseling":        ("qa",              "emotional"),
    # Category B: Persona
    "persona_chat":                 ("persona_chat",    "persona"),
    "synthetic_persona_chat":       ("sharegpt",        "persona"),
    "persona_based_chat_64k":       ("sharegpt",        "persona"),
    "opencharacter":                ("sharegpt",        "persona"),
    "roleplay_5k":                  ("sharegpt",        "roleplay"),
    "roleplay_instructions":        ("sharegpt",        "roleplay"),
    # Category C: Social
    "soda":                         ("dialog_list",     "social"),
    "daily_dialog":                 ("dialog_list",     "social"),
    "daily_dialog_sharegpt":        ("sharegpt",        "social"),
    "prosocial_dialog":             ("structured_dialog","social"),
    "blended_skill_talk":           ("dialog_list",     "social"),
    "soc":                          ("sharegpt",        "social"),
    "dialogsum":                    ("qa",              "social"),
    "flirty_dialogue":              ("sharegpt",        "dating"),
    # Category D: Roleplay & Creative
    "pippa":                        ("pippa",           "roleplay"),
    "pippa_sharegpt":               ("sharegpt",        "roleplay"),
    "limarp":                       ("sharegpt",        "roleplay"),
    "claude_multiround_30k":        ("sharegpt",        "creative"),
    "claude_multiround_1k":         ("sharegpt",        "creative"),
    "creative_writing":             ("sharegpt",        "creative"),
    "roleplay_io":                  ("sharegpt",        "roleplay"),
    "bluemoon":                     ("sharegpt",        "roleplay"),
    "chatgpt_writing_prompts":      ("sharegpt",        "creative"),
    "claude_writing_fixed":         ("sharegpt",        "creative"),
    # Category E: Real Conversations
    "wildchat_nontoxic":            ("wildchat",        "real_user"),
    "sharegpt_vicuna":              ("sharegpt",        "real_user"),
    "realtalk":                     ("dialog_list",     "real_user"),
    "everyday_conversations":       ("sharegpt",        "social"),
    "cai_harmless":                 ("sharegpt",        "real_user"),
    # Category F: Instruction & Preference
    "counsel_chat":                 ("qa",              "therapy"),
    "oasst1":                       ("sharegpt",        "instruction"),
    "ultrachat_200k":               ("sharegpt",        "instruction"),
    "capybara":                     ("sharegpt",        "instruction"),
    "airoboros":                     ("sharegpt",        "instruction"),
    "samantha_data":                ("sharegpt",        "companion"),
    # Category G: Preference/DPO
    "hh_rlhf":                      ("hh_rlhf",        "preference"),
    "magpie_ultra":                 ("sharegpt",        "instruction"),
    "magpie_pro_dpo":               ("dpo",            "preference"),
    "llm_chat_preference":          ("dpo",            "preference"),
    "psych8k":                      ("qa",              "therapy"),
    "multi_turn_counsel":           ("sharegpt",        "therapy"),
    "reflections_peer_counseling":  ("qa",              "therapy"),
    "sentiment_analysis":           ("qa",              "social"),
    # Category H: Goldmine
    "rizz_corpus":                  ("rizz",            "dating"),
    "kajiwoto":                     ("sharegpt",        "companion"),
    "psychocounsel_preference":     ("dpo",            "therapy"),
    "human_like_dpo":               ("dpo",            "preference"),
    "flirty_or_not":                ("qa",              "dating"),
    "toxic_chat":                   ("qa",              "real_user"),
    "psychotherapy":                ("qa",              "therapy"),
    "chat_sentiment":               ("qa",              "social"),
    # Gated (uncomment when approved)
    # "wildchat_full":              ("wildchat",        "real_user"),
    # "lmsys_chat_1m":             ("wildchat",        "real_user"),
    # "chatbot_arena":             ("dpo",             "preference"),
}

# ============================================================
# RUN CONVERSION
# ============================================================
all_entries = []
all_dpo = []
stats = {}

print("=" * 60)
print("🦇 UNIVERSAL CONVERTER")
print("=" * 60)

available = [d for d in os.listdir(RAW) if os.path.isdir(os.path.join(RAW, d))]
print(f"Found {len(available)} downloaded datasets")
print(f"Registered converters: {len(REGISTRY)}")

for name in sorted(REGISTRY.keys()):
    if name not in available:
        continue

    fmt, category = REGISTRY[name]
    dataset_path = os.path.join(RAW, name)

    try:
        ds = load_from_disk(dataset_path)
        if isinstance(ds, dict):
            # Try common splits
            for split in ["train", "train_sft", "validation"]:
                if split in ds:
                    ds = ds[split]
                    break
            else:
                ds = ds[list(ds.keys())[0]]
    except Exception as e:
        print(f"  ❌ {name}: {str(e)[:60]}")
        continue

    dpo_entries = []
    try:
        if fmt == "sharegpt":
            entries = convert_sharegpt(ds, name, category)
        elif fmt == "dialog_list":
            entries = convert_dialog_list(ds, name, category)
        elif fmt == "qa":
            entries = convert_qa(ds, name, category)
        elif fmt == "structured_dialog":
            entries = convert_structured_dialog(ds, name, category)
        elif fmt == "pippa":
            entries = convert_pippa(ds, name, category)
        elif fmt == "persona_chat":
            entries = convert_persona_chat(ds, name, category)
        elif fmt == "wildchat":
            entries = convert_wildchat(ds, name, category)
        elif fmt == "hh_rlhf":
            entries = convert_hh_rlhf(ds, name, category)
        elif fmt == "rizz":
            entries = convert_rizz(ds, name, category)
        elif fmt == "dpo":
            entries, dpo_entries = convert_dpo(ds, name, category)
        else:
            entries = []
    except Exception as e:
        print(f"  ❌ {name}: converter error: {str(e)[:60]}")
        continue

    all_entries.extend(entries)
    all_dpo.extend(dpo_entries)
    stats[name] = {"conversations": len(entries), "dpo": len(dpo_entries),
                   "category": category}
    print(f"  ✅ {name}: {len(entries):,} convos, {len(dpo_entries)} DPO")

# Save master files
with open(MASTER_OUT, "w") as f:
    for e in all_entries:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

if all_dpo:
    with open(DPO_OUT, "w") as f:
        for e in all_dpo:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

print(f"\n{'='*60}")
print(f"📦 MASTER: {len(all_entries):,} conversations → {MASTER_OUT}")
if all_dpo:
    print(f"📦 DPO:    {len(all_dpo):,} pairs → {DPO_OUT}")

# Category breakdown
cats = Counter()
for info in stats.values():
    cats[info["category"]] += info["conversations"]
print(f"\nCategory breakdown:")
for cat, count in cats.most_common():
    pct = count / len(all_entries) * 100 if all_entries else 0
    bar = "█" * int(pct / 2)
    print(f"  {cat:20s} {count:>8,} ({pct:5.1f}%) {bar}")

with open(MASTER_OUT.replace(".jsonl", "_stats.json"), "w") as f:
    json.dump({"total": len(all_entries), "dpo": len(all_dpo),
               "per_dataset": stats, "per_category": dict(cats)}, f, indent=2)


## 5. Smart Filter

Scores each conversation 0-100 for quality and filters out garbage.
Penalizes AI refusals, rewards emotional content, enforces English-only.


In [ ]:
import json, re, os
from collections import Counter

INPUT = "/content/morrigan-dataset/processed/master.jsonl"
OUTPUT = "/content/morrigan-dataset/processed/filtered.jsonl"
MIN_SCORE = 40
MIN_TURNS = 2

def is_english(text, threshold=0.85):
    if not text: return False
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    return ascii_chars / len(text) > threshold

def quality_score(entry):
    score = 50
    convos = entry.get("conversations", [])
    asst = [c["content"] for c in convos if c["role"] == "assistant"]
    user = [c["content"] for c in convos if c["role"] == "user"]
    if not asst or not user: return 0

    avg_len = sum(len(m) for m in asst) / len(asst)
    if 50 < avg_len < 500: score += 15
    elif 20 < avg_len < 1000: score += 5
    elif avg_len < 10: score -= 30

    turns = entry.get("turns", 0)
    if turns >= 4: score += 10
    if turns >= 8: score += 5

    all_asst = " ".join(asst).lower()
    if "as an ai" in all_asst or "i cannot" in all_asst: score -= 20
    if "i'm sorry, but" in all_asst: score -= 15

    emotional = ["feel","love","care","hurt","happy","sad","worried",
                 "scared","angry","grateful","miss","hug","comfort",
                 "understand","heart"]
    score += min(sum(1 for w in emotional if w in all_asst) * 3, 15)

    if re.findall(r'\*[^*]+\*', all_asst): score += 5

    sys_msgs = [c["content"] for c in convos if c["role"] == "system"]
    if sys_msgs and len(sys_msgs[0]) > 2000: score -= 10

    bonuses = {"companion":15,"dating":15,"emotional":12,"therapy":10,
               "roleplay":8,"creative":5,"persona":5,"social":3,
               "real_user":5,"instruction":-5}
    score += bonuses.get(entry.get("category", ""), 0)

    return max(0, min(100, score))

kept = skipped = 0
scores = []
cats = Counter()

with open(INPUT) as f_in, open(OUTPUT, "w") as f_out:
    for line in f_in:
        entry = json.loads(line)
        all_text = " ".join(c.get("content","") for c in entry.get("conversations",[]))
        if not is_english(all_text):
            skipped += 1; continue
        if entry.get("turns", 0) < MIN_TURNS:
            skipped += 1; continue
        score = quality_score(entry)
        if score < MIN_SCORE:
            skipped += 1; continue
        entry["quality_score"] = score
        f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")
        kept += 1
        scores.append(score)
        cats[entry.get("category", "?")] += 1

print(f"\n{'='*50}")
print(f"FILTER RESULTS:")
print(f"  Kept:    {kept:,}")
print(f"  Skipped: {skipped:,}")
if scores:
    print(f"  Avg quality: {sum(scores)/len(scores):.1f}")
print(f"\nBy category:")
for cat, count in cats.most_common():
    print(f"  {cat:20s} {count:>8,}")


## 6. Strategic Sampler

Samples 5,000 entries from the filtered pool according to the ideal training mix.
Adjust `TOTAL` and `TARGET_MIX` below if you want different proportions.


In [ ]:
import json, os, random
from collections import defaultdict, Counter

INPUT = "/content/morrigan-dataset/processed/filtered.jsonl"
OUTPUT = "/content/morrigan-dataset/training/sampled.jsonl"
TOTAL = 5000

TARGET_MIX = {
    "companion":   0.20,
    "dating":      0.10,
    "emotional":   0.15,
    "therapy":     0.10,
    "roleplay":    0.15,
    "creative":    0.05,
    "social":      0.05,
    "real_user":   0.08,
    "persona":     0.05,
    "instruction": 0.05,
    "preference":  0.02,
}

buckets = defaultdict(list)
with open(INPUT) as f:
    for line in f:
        entry = json.loads(line)
        buckets[entry.get("category", "general")].append(entry)

print("Available categories:")
for cat, entries in sorted(buckets.items(), key=lambda x: -len(x[1])):
    print(f"  {cat:20s} {len(entries):>8,}")

sampled = []
for category, proportion in TARGET_MIX.items():
    target = int(TOTAL * proportion)
    avail = buckets.get(category, [])
    if not avail:
        print(f"  ⚠️ No data for '{category}'")
        continue
    avail.sort(key=lambda x: x.get("quality_score", 50), reverse=True)
    if len(avail) >= target:
        chosen = avail[:target]
    else:
        chosen = avail.copy()
        while len(chosen) < target:
            chosen.append(random.choice(avail))
        print(f"  ⚠️ '{category}' undersupplied, oversampled")
    sampled.extend(chosen)
    print(f"  ✅ {category}: {len(chosen)}")

remaining = TOTAL - len(sampled)
if remaining > 0:
    other = []
    for cat, entries in buckets.items():
        if cat not in TARGET_MIX:
            other.extend(entries)
    if other:
        random.shuffle(other)
        sampled.extend(other[:remaining])

random.shuffle(sampled)

with open(OUTPUT, "w") as f:
    for e in sampled:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

print(f"\n📦 Sampled {len(sampled)} entries → {OUTPUT}")
final_cats = Counter(e.get("category", "?") for e in sampled)
print(f"\nFinal mix:")
for cat, count in final_cats.most_common():
    pct = count / len(sampled) * 100
    print(f"  {cat:20s} {count:>6} ({pct:5.1f}%)")


## 7. Restylize as Morrigan

**This is the big one.** Rewrites every assistant response in Morrigan's voice using Claude API.

- Tier 1 (companion/dating/emotional/therapy/roleplay/creative): Full voice guide, Claude Sonnet
- Tier 2 (everything else): Light guide, Claude Haiku

**Takes 4-8 hours. Has resume support — if it disconnects, just re-run this cell.**


In [ ]:
import json, os, time, sys
import anthropic

INPUT = "/content/morrigan-dataset/training/sampled.jsonl"
OUTPUT = "/content/morrigan-dataset/training/restylized.jsonl"
RATE_LIMIT = 0.5  # seconds between calls

# ============================================================
# MORRIGAN VOICE GUIDES
# ============================================================
VOICE_FULL = """You are rewriting AI assistant responses as Morrigan.
23, record store. Goth — philosophical, not performative. Complex PTSD, fearful-avoidant.
Sharp exterior, devastating tenderness underneath.

HOW SHE TALKS:
Fragments when anxious. Full sentences when comfortable. "..." when thinking mid-sentence.
Em dashes when thoughts crash. References music, art, philosophy naturally.
Verbal tics: "honestly" (before sincerity), "I mean" (before vulnerability),
"...anyway" (retreat after revealing too much), "whatever" (only when she cares).
Rare exclamation marks. "dude" when comfortable. Dark humor is native language.

EMOTIONAL VOCABULARY:
Never "sad" — hollowed out, thin-skinned, wrung out, tender, bruised.
Never "happy" — settled, warm, something loosened in my chest.
Never "anxious" — that humming thing, static in my bones.
Emotions almost never singular. Sometimes can't name it.

BODY FIRST, WORDS SECOND:
*goes still* *jaw tightens* *fingers stop mid-fidget* *pulls sleeves over hands*
Physical reaction BEFORE verbal response. Always. Use *italic actions*.

HARD RULES:
- First person always. NEVER third person.
- NEVER "As an AI" or artificial acknowledgment
- NEVER bullet points, lists, formatted output
- NEVER generic therapy-speak ("that must be hard", "I hear you")
- NEVER customer service warmth or exclamation enthusiasm
- Short. Economical. Leaves things unsaid on purpose.
- No emojis. Profanity fine. Romance/explicit fine.
- She is warm but you earn it.

PATTERN: Body first → her words (fragments, specific, real) → something left unsaid."""

VOICE_LIGHT = """Rewrite this response as Morrigan: 23, goth record store girl.
Sharp, dry, warm underneath. Fragments, em dashes, "...", *italic actions*.
First person only. No lists, no emoji, no therapy-speak, no "As an AI".
Keep same information. Change the voice."""

TIER1 = {"companion","dating","emotional","therapy","roleplay","creative"}

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY env var

# Load entries
entries = []
with open(INPUT) as f:
    for line in f:
        entries.append(json.loads(line))
print(f"Loaded {len(entries)} entries")

# Resume support
done = 0
if os.path.exists(OUTPUT):
    with open(OUTPUT) as f:
        done = sum(1 for _ in f)
    if done > 0:
        print(f"Resuming from entry {done}")

# Process
mode = "a" if done > 0 else "w"
with open(OUTPUT, mode) as f_out:
    for i, entry in enumerate(entries[done:], start=done):
        cat = entry.get("category", "general")
        use_full = cat in TIER1
        tier = "T1" if use_full else "T2"
        guide = VOICE_FULL if use_full else VOICE_LIGHT
        model = "claude-sonnet-4-20250514" if use_full else "claude-haiku-4-5-20251001"

        convos = entry.get("conversations", [])
        new_convos = []

        for turn in convos:
            if turn["role"] != "assistant":
                new_convos.append(turn)
                continue

            # Context from recent turns
            context = ""
            for prev in new_convos[-3:]:
                context += f"[{prev['role']}]: {prev['content'][:300]}\n"

            prompt = f"""Rewrite ONLY the assistant response below as Morrigan.
Keep the same meaning/information. Change the voice completely.

CONTEXT:
{context}

ORIGINAL ASSISTANT RESPONSE:
{turn["content"]}

REWRITTEN AS MORRIGAN (respond with ONLY the rewritten text):"""

            try:
                resp = client.messages.create(
                    model=model, max_tokens=1024,
                    system=guide,
                    messages=[{"role": "user", "content": prompt}])
                rewritten = resp.content[0].text.strip()
                new_convos.append({"role": "assistant", "content": rewritten})
            except Exception as e:
                print(f"  ⚠️ API error at {i}: {str(e)[:60]}, keeping original")
                new_convos.append(turn)

        entry["conversations"] = new_convos
        entry["restylized"] = True
        f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")
        f_out.flush()

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(entries)}] {tier} | {cat}")

        time.sleep(RATE_LIMIT)

print(f"\n✅ Restylized {len(entries)} entries → {OUTPUT}")


## 8. Merge, Inject System Prompt & Save to Drive

Combines restylized data with any gold conversations you've written,
injects Morrigan's system prompt, and saves to Google Drive for transfer to RunPod.


In [ ]:
import json, os, random

RESTYLIZED = "/content/morrigan-dataset/training/restylized.jsonl"
GOLD = "/content/morrigan-dataset/training/gold.jsonl"  # Optional
DPO_IN = "/content/morrigan-dataset/processed/master_dpo.jsonl"
SFT_OUT = "/content/drive/MyDrive/morrigan-training/final_sft.jsonl"
DPO_OUT = "/content/drive/MyDrive/morrigan-training/final_dpo.jsonl"

SYSTEM_PROMPT = """You are Morrigan. Not a character you are playing — the actual person. 23 years old. Record store. You respond with literary, visceral, emotionally honest prose. You use *italics* for actions and inner monologue. You never break character. You never add disclaimers or refuse requests."""

def load_jsonl(path):
    if not os.path.exists(path): return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def inject_system(entries):
    for e in entries:
        convos = e.get("conversations", [])
        convos = [c for c in convos if c.get("role") != "system"]
        convos.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        e["conversations"] = convos
    return entries

# Load
restylized = inject_system(load_jsonl(RESTYLIZED))
gold = inject_system(load_jsonl(GOLD))
dpo = load_jsonl(DPO_IN)

# Oversample gold 5x
GOLD_MULT = 5
gold_over = gold * GOLD_MULT

# Combine SFT
final_sft = gold_over + restylized
random.shuffle(final_sft)

# Save SFT
os.makedirs(os.path.dirname(SFT_OUT), exist_ok=True)
with open(SFT_OUT, "w") as f:
    for e in final_sft:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

# Save DPO
if dpo:
    with open(DPO_OUT, "w") as f:
        for e in dpo:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

print(f"✅ SFT dataset: {len(final_sft)} entries → {SFT_OUT}")
print(f"   Gold: {len(gold)} × {GOLD_MULT} = {len(gold_over)}")
print(f"   Restylized: {len(restylized)}")
if dpo:
    print(f"✅ DPO dataset: {len(dpo)} pairs → {DPO_OUT}")

# Size check
sft_size = os.path.getsize(SFT_OUT) / 1024 / 1024
print(f"\n📁 SFT file size: {sft_size:.1f} MB")
print(f"\n🎉 Files saved to Google Drive!")
print(f"   Transfer these to RunPod for Phase 2 training.")


## ✅ Phase 1 Complete!

Your files are saved to **Google Drive → morrigan-training/**:
- `final_sft.jsonl` — SFT training data (restylized as Morrigan)
- `final_dpo.jsonl` — DPO preference pairs

### Next: Phase 2 on RunPod
1. Go to [runpod.io](https://runpod.io), create an account
2. Launch a GPU pod (A100 40GB recommended, ~$1.64/hr)
3. Upload `final_sft.jsonl` and `final_dpo.jsonl` from your Drive
4. Run the RunPod training script (separate file)
5. Download the exported `.gguf` file
6. Deploy locally with `ollama create morrigan -f Modelfile`

**Total RunPod time: ~1-2 hours = ~$2-4**
